In [ ]:
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
import numpy as np
import pandas as pd


In [ ]:
# Simulate fold results for ablation demo (replace with actual trained model outputs)
from src.evaluation.metrics import compute_metrics, bootstrap_ci, aggregate_fold_results

np.random.seed(42)
models = {
    'mobilenet_v3_large': {'y_true': np.random.randint(0,2,100), 'y_prob': np.random.rand(100)},
    'efficientnet_b0':    {'y_true': np.random.randint(0,2,100), 'y_prob': np.random.rand(100)},
    'resnet50':           {'y_true': np.random.randint(0,2,100), 'y_prob': np.random.rand(100)},
    'fusion_intermediate':{'y_true': np.random.randint(0,2,100), 'y_prob': np.random.rand(100)},
    'fusion_early':       {'y_true': np.random.randint(0,2,100), 'y_prob': np.random.rand(100)},
    'radiomics_xgb':      {'y_true': np.random.randint(0,2,100), 'y_prob': np.random.rand(100)},
}

for name in models:
    models[name]['y_true'] = np.random.randint(0, 2, 100)
    models[name]['y_prob'] = np.clip(np.random.rand(100), 0.01, 0.99)


In [ ]:
# Build ablation table with DeLong tests
from src.evaluation.statistical_tests import build_ablation_table

ablation_df = build_ablation_table(models, reference_model='fusion_intermediate')
print(ablation_df[['model', 'auc', 'auc_ci_95', 'sensitivity', 'specificity',
                    'delta_auc_vs_ref', 'delong_p', 'significant']].to_string(index=False))
ablation_df.to_csv('/content/ablation_table.csv', index=False)


In [ ]:
# Calibration plots
import matplotlib.pyplot as plt
from src.evaluation.metrics import build_calibration_data

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, res) in zip(axes.flatten(), models.items()):
    cal_df = build_calibration_data(res['y_true'], res['y_prob'])
    ax.plot(cal_df['mean_predicted'], cal_df['fraction_positive'], 's-', label=name)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set_title(name)
    ax.set_xlabel('Mean Predicted')
    ax.set_ylabel('Fraction Positive')
    ax.legend(fontsize=8)
plt.suptitle('Calibration Curves', fontsize=14)
plt.tight_layout()
plt.savefig('/content/calibration_curves.png', dpi=150)
plt.show()
